In [5]:
import pandas as pd

# ============================================================
# LOAD DATASET
# ============================================================
df = pd.read_csv('../dataset/final_dataset_raw.csv', sep=';')

print(f"=== DATASET INFO ===")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# ============================================================
# FIX DATA TYPES
# ============================================================

# Fix transaction_value — European format (comma as decimal)
df['transaction_value'] = (
    df['transaction_value']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# Fix timestamp
df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s')

# Fix mint_timestamp (has missing values)
df['mint_timestamp'] = pd.to_numeric(
    df['mint_timestamp'], errors='coerce'
)
df['mint_timestamp_dt'] = pd.to_datetime(
    df['mint_timestamp'], unit='s', errors='coerce'
)

print(f"\n=== AFTER FIX DTYPES ===")
print(df[['transaction_value',
          'timestamp_dt',
          'mint_timestamp_dt']].dtypes)

# ============================================================
# CHECK TRANSACTION VALUE
# ============================================================
print(f"\n=== TRANSACTION VALUE ===")
print(f"Dtype     : {df['transaction_value'].dtype}")
print(f"Zero value: {(df['transaction_value']==0).sum():,} "
      f"({(df['transaction_value']==0).mean():.1%})")
print(f"Non-zero  : {(df['transaction_value']>0).sum():,} "
      f"({(df['transaction_value']>0).mean():.1%})")
print(f"Min: {df['transaction_value'].min()}")
print(f"Max: {df['transaction_value'].max()}")

# ============================================================
# CHECK DATE RANGE
# ============================================================
print(f"\n=== DATE RANGE ===")
print(f"From: {df['timestamp_dt'].min()}")
print(f"To  : {df['timestamp_dt'].max()}")
print(f"Duration: {(df['timestamp_dt'].max() - df['timestamp_dt'].min()).days} days")

# Per month
monthly = df.groupby(
    df['timestamp_dt'].dt.to_period('M')
).size()
print(f"\n=== PER MONTH ===")
print(monthly)

# ============================================================
# CHECK MINT TIMESTAMP
# ============================================================
print(f"\n=== MINT TIMESTAMP ===")
print(f"Missing  : {df['mint_timestamp'].isnull().sum():,} "
      f"({df['mint_timestamp'].isnull().mean():.1%})")
print(f"Has value: {df['mint_timestamp'].notna().sum():,} "
      f"({df['mint_timestamp'].notna().mean():.1%})")

# ============================================================
# SAMPLE DATA
# ============================================================
print(f"\n=== SAMPLE (first 3 rows) ===")
print(df[['timestamp_dt', 'nft_address', 'token_id',
          'from_address', 'to_address',
          'transaction_value', 'mint_timestamp_dt',
          'num_transitions']].head(3).to_string())

=== DATASET INFO ===
Shape: (1048575, 14)
Columns: ['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions']

Dtypes:
transaction_hash          str
block_number            int64
timestamp               int64
nft_address               str
token_id                  str
from_address              str
to_address                str
transaction_value         str
mint_timestamp        float64
transfers_out_from      int64
transfers_in_from       int64
transfers_out_to        int64
transfers_in_to         int64
num_transitions         int64
dtype: object

Missing values:
transaction_hash          0
block_number              0
timestamp                 0
nft_address               0
token_id                  0
from_address              0
to_address                0
transaction_value         0
mint_timestamp    

In [6]:
# ============================================================
# FIX ALL DATA ISSUES
# ============================================================

print("Fixing data issues...")

# Fix 1: token_id — convert scientific notation string to int
# "6,68293E+76" → replace comma → float → handle carefully
# Note: these are very large numbers (ERC-1155 token IDs)
df['token_id_clean'] = (
    df['token_id']
    .astype(str)
    .str.replace(',', '.', regex=False)  # fix decimal separator
)

# Check if conversion works
print(f"Sample token_id before: {df['token_id'].head(3).tolist()}")
print(f"Sample token_id after : {df['token_id_clean'].head(3).tolist()}")

# Fix 2: transaction_value outlier
# Max ETH supply ≈ 120 million ETH = 1.2e+26 Wei
# Anything above that is likely data error
MAX_REASONABLE_VALUE = 1e+26
outliers = (df['transaction_value'] > MAX_REASONABLE_VALUE).sum()
print(f"\nOutlier transaction values: {outliers:,}")

df['transaction_value_clean'] = df['transaction_value'].clip(
    upper=MAX_REASONABLE_VALUE
)

# Fix 3: filter sales only (value > 0)
# Aligned with Liu et al. (2023)
sales = df[df['transaction_value'] > 0].copy()
print(f"\nSales only (value > 0): {len(sales):,} "
      f"({len(sales)/len(df):.1%})")

# Fix 4: remove burn addresses
BURN_ADDRESSES = {
    '0x0000000000000000000000000000000000000000',
    '0x000000000000000000000000000000000000dead'
}
burn_lower = {a.lower() for a in BURN_ADDRESSES}

sales_clean = sales[
    ~sales['from_address'].str.lower().isin(burn_lower) &
    ~sales['to_address'].str.lower().isin(burn_lower)
].copy().reset_index(drop=True)

print(f"After burn filter: {len(sales_clean):,}")

# ============================================================
# CHECK TEMPORAL SPLIT FEASIBILITY
# ============================================================
print(f"\n=== TEMPORAL SPLIT PLAN ===")
print(f"Total period: {sales_clean['timestamp_dt'].min()} "
      f"to {sales_clean['timestamp_dt'].max()}")

# Split: train = Aug 1-16, test = Aug 17-24
split_date = pd.Timestamp('2021-08-17')
train = sales_clean[sales_clean['timestamp_dt'] < split_date]
test  = sales_clean[sales_clean['timestamp_dt'] >= split_date]

print(f"Train (Aug 1-16) : {len(train):,} rows "
      f"({len(train)/len(sales_clean):.1%})")
print(f"Test  (Aug 17-24): {len(test):,} rows "
      f"({len(test)/len(sales_clean):.1%})")

# ============================================================
# FINAL CLEAN DATASET SUMMARY
# ============================================================
print(f"\n=== CLEAN DATASET SUMMARY ===")
print(f"Total rows    : {len(sales_clean):,}")
print(f"Unique wallets: "
      f"{pd.concat([sales_clean['from_address'], sales_clean['to_address']]).nunique():,}")
print(f"Unique NFTs   : {sales_clean['nft_address'].nunique():,}")
print(f"Date range    : {sales_clean['timestamp_dt'].min()} "
      f"to {sales_clean['timestamp_dt'].max()}")
print(f"Missing mint  : {sales_clean['mint_timestamp'].isnull().sum():,} "
      f"({sales_clean['mint_timestamp'].isnull().mean():.1%})")

Fixing data issues...
Sample token_id before: ['6,68293E+76', '1,19851E+76', '1,54122E+76']
Sample token_id after : ['6.68293E+76', '1.19851E+76', '1.54122E+76']

Outlier transaction values: 17,704

Sales only (value > 0): 728,152 (69.4%)
After burn filter: 722,814

=== TEMPORAL SPLIT PLAN ===
Total period: 2021-08-01 00:00:17 to 2021-08-24 19:39:35
Train (Aug 1-16) : 471,312 rows (65.2%)
Test  (Aug 17-24): 251,502 rows (34.8%)

=== CLEAN DATASET SUMMARY ===
Total rows    : 722,814
Unique wallets: 118,528
Unique NFTs   : 1,560
Date range    : 2021-08-01 00:00:17 to 2021-08-24 19:39:35
Missing mint  : 39,406 (5.5%)
